# Our project workflow
1. Install and configure Apache Spark
2. Extract – Read the CSV dataset
3. Explore – Understand the dataset
4. Clean – Remove duplicates and separate invalid records
5. Transform – Convert dates and calculate sales
6. Validate – Check data quality
7. Load – Save the processed dataset

In [ ]:
# Install Apache Spark Python library
!pip install pyspark

# pip – Python's package installer.
# install – Tells Python to install a library.
# pyspark – Python API for Apache Spark.
# ! – Tells Google Colab to execute a system command.
#Google Colab provides Python, but PySpark may not be installed by default. We install it so that we can use Apache Spark through Python.

Step 2. Import the required libraries

In [ ]:
# Import SparkSession to start Apache Spark
from pyspark.sql import SparkSession

# Import functions required for data processing
from pyspark.sql.functions import col, when, to_timestamp, sum

# SparkSession	Creates the entry point for Spark
# col	Refers to a DataFrame column
# when	Applies conditional logic
# to_timestamp	Converts text into timestamp format
# sum	Calculates the sum of values

# We'll use these functions in our cleaning and transformation stages.

Step 3. Create a Spark Session. Start Apache Spark

In [ ]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("Online Retail ETL Project") \
    .master("local[*]") \
    .getOrCreate()

# Display Spark version
print("Spark Version:", spark.version)

# Why do we need this?

# Before processing data with Spark, we need to start a Spark session.
# - builder – Starts configuring the Spark session.
# - appName() – Gives our project a name.
# - master("local[*]") – Runs Spark locally using the available CPU cores.
# - getOrCreate() – Creates a new session or reuses an existing one.
# - spark.version – Displays the installed Spark version.

Spark Version: 4.0.5


# Phase 2: Extracting the dataset using PySpark In our Pandas project, we used pd.read_csv() to read the CSV file. In PySpark, we'll use spark.read.csv()

Step 4. Upload the CSV file to Google Colab

In [ ]:
# Import the file upload function
from google.colab import files

# Upload the Online Retail CSV file
uploaded = files.upload()

# Why are we doing this?
# Our dataset is currently stored on your computer. We need to upload it to Colab so Spark can access and process it.

Saving Online Retail.csv to Online Retail (2).csv


Step 5. Read the CSV file using PySpark. Extract data

In [ ]:
# Read the CSV file into a Spark DataFrame
df = spark.read.csv(
    "Online Retail.csv",
    header=True,
    inferSchema=True
)

# Display the first five records
df.show(5)

# spark.read	Starts reading a dataset
# .csv()	Specifies that the file is CSV
# header=True	Uses the first row as column names
# inferSchema=True	Automatically identifies data types
# df.show(5)	Displays five records
#A Spark DataFrame is different from a Pandas DataFrame.
#Spark processes data using its distributed processing engine and generally evaluates transformations lazily.

+---------+---------+--------------------+--------+------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity| InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/10 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/10 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/10 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/10 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/10 8:26|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+------------+---------+----------+--------------+
only showing top 5 rows


Step 6. Check the number of rows and columns
Dataset dimensions

In [ ]:
# Count the total number of records
total_rows = df.count()

# Count the total number of columns
total_columns = len(df.columns)

# Display the dataset dimensions
print("Total Rows:", total_rows)
print("Total Columns:", total_columns)

Total Rows: 541909
Total Columns: 8


Step 7. Check the structure and data types
Print schema

In [ ]:
# Display the structure and data types of the dataset
df.printSchema()

# This tells us whether each column is stored as a string, integer, double, or
#another data type.
#In Apache Spark (and PySpark), printSchema() reads the internal StructType
#metadata stored on a DataFrame and prints each column's name, data type, and nullability in a readable tree format.
# In Apache Spark, a schema is the blueprint that defines the structure of a DataFrame.
# It specifies the exact column names, their data types (such as integers, strings, or dates), and whether each column can hold null values

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)



Step 8. Check missing values

In [ ]:
# Import the required functions
from pyspark.sql.functions import col, sum

# Count missing values in every column
missing_values = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

# Display missing value counts
missing_values.show()

# Step 1: col(c) current column select karega.

# Step 2: .isNull() check karega ki value missing hai ya nahi.

# Step 3: .cast("int") True/False ko 1/0 mein badlega.

# Step 4: sum() un sabhi 1s ko add karega.

# Matlab sum() bahar isliye hai kyunki uska kaam andar ke poore calculation ka result add karna hai.

# Agar sum() ko pehle laga dete aur uske andar col(c) nahi hota, toh Spark ko ye nahi pata chalta ki kis column ki missing values count karni hain.


# - isNull() checks whether a value is missing.
# - cast("int") converts True/False into 1/0.
# - sum() adds the missing-value indicators.
# - alias(c) gives the result the original column name.
# - select() creates a result containing the missing-value counts.

# Iska matlab hai ki hum PySpark ki functions library se do specific functions import kar rahe hain:
# - col – kisi particular column ko refer karne ke liye.
# - sum – values ka total calculate karne ke liye.
# Inhe import karna kyun zaroori hai?

# PySpark ko batana hota hai ki hum column par kaunsa operation perform karna chahte hain. col() aur sum() uske built-in functions hain.

# Ek important baat: yahan sum PySpark ka function hai, Python ka normal sum() nahi.

+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|        0|        0|       1454|       0|          0|        0|    135080|      0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+



# Phase 3: Data Exploration using PySpark
Ab tak humne:
- PySpark install kiya.
- Spark Session create kiya.
- CSV upload aur read ki.
- Rows aur columns count kiye.
- Schema check kiya.
- Missing values identify ki.

Ab hum dataset ki quality check karenge, bilkul waise hi jaise humne Pandas project mein kiya tha.

Step 9. Count duplicate records

In [ ]:
# Count duplicate records in the dataset
duplicate_count = df.count() - df.dropDuplicates().count()

# Display the duplicate count
print("Total Duplicate Records:", duplicate_count)

# Yahan do operations ho rahe hain:
# 1. df.count() – Original DataFrame mein total records count karta hai.
# 2. df.dropDuplicates().count() – Duplicate records remove karne ke baad total records count karta hai.
# Dono ka difference batata hai ki kitne duplicate records the.

# Example:
# - Original records = 1000
# - Unique records = 950
# - Duplicate records = 50

Total Duplicate Records: 5268


Step 10. Identify cancelled transactions
Hamare Online Retail dataset mein cancelled invoices ki identification C se start hone wale InvoiceNo se hoti hai.

In [ ]:
# Identify cancelled transactions
df_cancelled = df.filter(
    col("InvoiceNo").startswith("C")
)

# Display the number of cancelled transactions
print("Cancelled Transactions:", df_cancelled.count())

# filter() – Conditions ke according records select karta hai.
# col("InvoiceNo") – InvoiceNo column select karta hai.
# startswith("C") – Check karta hai ki invoice number C se start ho raha hai ya nahi.
# count() – Matching records count karta hai.

Cancelled Transactions: 9288


Step 11. Identify negative quantities (Returns)
Negative quantity ka matlab ho sakta hai ki customer ne product return kiya hai.

In [ ]:
# Identify transactions with negative quantities
df_returns = df.filter(
    col("Quantity") < 0
)

# Display the number of return records
print("Return Records:", df_returns.count())


Return Records: 10624


Step 12. Identify invalid prices
Zero ya negative UnitPrice ko hum questionable price records maanenge.

In [ ]:
# Identify records with zero or negative prices
df_invalid_prices = df.filter(
    col("UnitPrice") <= 0
)

# Display the number of invalid price records
print("Invalid Price Records:", df_invalid_prices.count())

Invalid Price Records: 2517


Identify return records

In [ ]:
# Identify transactions with negative quantities
df_returns = df.filter(
    col("Quantity") < 0
)

# Count return records
print("Return Records:", df_returns.count())


# df.filter() → Condition ke according rows select karta hai.
# col("Quantity") → Quantity column ko select karta hai.
# < 0 → Check karta hai ki quantity zero se kam hai ya nahi.
# df_returns → Aise records ko alag DataFrame mein store karta hai.
# .count() → Total return records count karta hai.

Return Records: 10624


# Phase 4: Data Cleaning using PySpark

Create a separate working DataFrame
Sabse pehle hum original dataset ki ek separate working reference banayenge

In [ ]:
# Create a working DataFrame from the original dataset
df_clean = df

# Display the total number of records
print("Original Records:", df.count())
print("Working Records:", df_clean.count())

# 1. df_clean = df
# Iska matlab hai ki hum original DataFrame df ko df_clean naam se refer kar rahe hain.
# Ab hum cleaning operations df_clean par perform karenge.
# 2. df.count()
# Original dataset mein total records count karega.
# 3. df_clean.count()
# Working DataFrame mein total records count karega.
# Dono ki count initially same honi chahiye.
# Important concept: Spark DataFrames are immutable.
# Spark mein hum existing DataFrame ko directly modify nahi karte. Jab transformation karte hain, toh result ko kisi naye variable mein store karte hain.
# Example: df_clean = df_clean.dropDuplicates()

Original Records: 541909
Working Records: 541909


Remove duplicates

In [ ]:
# Count duplicate records before cleaning
duplicates_before = df_clean.count() - df_clean.dropDuplicates().count()
#Cleaning se pehle duplicate records count kar rahe hain.

# Remove duplicate records
df_clean = df_clean.dropDuplicates()
#Duplicate rows remove karke updated DataFrame df_clean mein store kar rahe hain.

# Count duplicate records after cleaning
duplicates_after = df_clean.count() - df_clean.dropDuplicates().count()
#Check kar rahe hain ki cleaning ke baad kitne duplicates bache hain.

# Display cleaning results
print("Duplicates Before Cleaning:", duplicates_before)
print("Duplicates After Cleaning:", duplicates_after)
print("Records After Cleaning:", df_clean.count())

# dropDuplicates() – Exact duplicate rows remove karta hai.
# count() – Total rows count karta hai.
# df_clean = – Updated DataFrame ko save karta hai.

Duplicates Before Cleaning: 5268
Duplicates After Cleaning: 0
Records After Cleaning: 536641


Cancelled Transactions ko separate karenge

In [ ]:
# Identify cancelled invoices
cancellation_condition = col("InvoiceNo").startswith("C")
#InvoiceNo check kar rahe hain ki woh C se start hota hai ya nahi.

# Store cancelled transactions separately
df_cancelled = df_clean.filter(cancellation_condition)
#Jo invoices C se start hote hain, unhe df_cancelled mein store karenge.

# Keep non-cancelled transactions in the main dataset
df_sales = df_clean.filter(~cancellation_condition)
#Yahan ~ ka matlab NOT hai.
#Matlab jo invoices C se start nahi hote, unhe df_sales mein rakhenge.

# Display the number of records
print("Cancelled Records:", df_cancelled.count())
print("Non-Cancelled Records:", df_sales.count())

Cancelled Records: 9251
Non-Cancelled Records: 527390


Return Transactions ko Sales se Separate karenge.

In [ ]:
# Identify transactions with negative quantities
returns_condition = col("Quantity") < 0
#Quantity zero se kam hai ya nahi, ye check karne ki condition banayi.

# Store return transactions separately
df_returns = df_sales.filter(returns_condition)
#Negative quantity wale records ko df_returns mein store kiya.

# Store records with zero quantity separately
df_zero_quantity = df_sales.filter(col("Quantity") == 0)
#Zero quantity wale records ko alag store kiya, taaki unka data lose na ho.

# Keep only positive quantity transactions as sales
df_sales = df_sales.filter(col("Quantity") > 0)
#Ab main sales dataset mein sirf positive quantity wale records rakhe.

# Display the results
print("Return Records:", df_returns.count())
print("Zero Quantity Records:", df_zero_quantity.count())
print("Valid Sales Records:", df_sales.count())

Return Records: 1336
Zero Quantity Records: 0
Valid Sales Records: 526054


Separate invalid price records

In [ ]:
# Identify records with zero or negative prices
invalid_price_condition = col("UnitPrice") <= 0

# Store invalid price records separately
df_invalid_sales_prices = df_sales.filter(invalid_price_condition)

# Keep only records with positive prices
df_sales = df_sales.filter(col("UnitPrice") > 0)

# Display the results
print("Invalid Price Records:", df_invalid_sales_prices.count())
print("Valid Sales Records:", df_sales.count())

Invalid Price Records: 1176
Valid Sales Records: 524878


Convert InvoiceDate into Timestamp

In [ ]:
# Rebuild the valid sales dataset
df_sales = df_clean.filter(
    (~col("InvoiceNo").startswith("C")) &
    (col("Quantity") > 0) &
    (col("UnitPrice") > 0)
)

# Iska matlab hai ki hum df_clean DataFrame se kuch conditions ke according records select karenge aur result ko df_sales mein store karenge.
# - df_clean = Duplicate removal ke baad wala dataset.
# - filter() = Condition ke according rows select karta hai.
# - df_sales = Hamara final valid sales dataset.
# col("InvoiceNo")	InvoiceNo column select karo
# .startswith("C")	Check karo invoice C se start hota hai ya nahi
# ~	NOT, yani opposite condition
# col("Quantity") = Quantity column select karta hai.
# > = Greater than.
# 0 = Zero.
#& Ye check karta hai ki product ka price zero se greater hai ya nahi.
#& ka matlab hai AND. Yani ek record ko df_sales mein tabhi rakha jayega jab teeno conditions TRUE hongi.

# Check the InvoiceDate data type
df_sales.printSchema()
#Iska kaam hai DataFrame ke columns aur unke data types display karna.

# Display the first five invoice dates
df_sales.select("InvoiceDate").show(5, truncate=False)
#select("InvoiceDate") = Sirf InvoiceDate column select karo.
# show(5) = First five records dikhao.
# truncate=False = Values ko short karke display mat karo.

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+-------------+
|InvoiceDate  |
+-------------+
|12/1/10 10:52|
|12/1/10 11:33|
|12/1/10 11:41|
|12/1/10 12:43|
|12/1/10 12:49|
+-------------+
only showing top 5 rows


# Transformation step

Calculate TotalSales

# Data Analysis

In [ ]:
# Calculate total sales for each transaction
df_sales = df_sales.withColumn(
    "TotalSales",  # Ye naye column ka naam hai jo hum create kar rahe hain.
    col("Quantity") * col("UnitPrice")
)

# - col("Quantity") – Quantity column select karta hai.
# - * – Multiplication operator hai.
# - col("UnitPrice") – UnitPrice column select karta hai.
# Dono columns ki values multiply hongi aur result TotalSales mein store hoga.

# Display the first five records
df_sales.show(5)
# Updated DataFrame ke first five records display karega.

+---------+---------+--------------------+--------+-------------+---------+----------+--------------+------------------+
|InvoiceNo|StockCode|         Description|Quantity|  InvoiceDate|UnitPrice|CustomerID|       Country|        TotalSales|
+---------+---------+--------------------+--------+-------------+---------+----------+--------------+------------------+
|   536398|    22112|CHOCOLATE HOT WAT...|       9|12/1/10 10:52|     4.95|     13448|United Kingdom|44.550000000000004|
|   536406|    82486|WOOD S/3 CABINET ...|       4|12/1/10 11:33|     6.95|     17850|United Kingdom|              27.8|
|   536408|    21832|CHOCOLATE CALCULATOR|      12|12/1/10 11:41|     1.65|     14307|United Kingdom|19.799999999999997|
|   536520|    21584|RETROSPOT SMALL T...|      12|12/1/10 12:43|     1.65|     14729|United Kingdom|19.799999999999997|
|   536522|    22151|PLACE SETTING WHI...|       2|12/1/10 12:49|     0.42|     15012|United Kingdom|              0.84|
+---------+---------+-----------

Average Sales

In [ ]:
# Import the average function
from pyspark.sql.functions import avg

# Calculate average sales per transaction line
average_sales = df_sales.select(
    avg("TotalSales").alias("AverageSales")
)

# Display the average sales
average_sales.show()

+------------------+
|      AverageSales|
+------------------+
|20.275398862211457|
+------------------+



Calculate Total Business Revenue karenge.

In [ ]:
# Import the sum function
from pyspark.sql.functions import sum

# Calculate total business revenue
total_revenue = df_sales.select(
    sum("TotalSales").alias("TotalRevenue")
)

# Display total revenue
total_revenue.show()

+--------------------+
|        TotalRevenue|
+--------------------+
|1.0642110803999824E7|
+--------------------+



Country-wise Sales Analysis karenge.

In [ ]:
# Import the required functions
from pyspark.sql.functions import sum, col

# Calculate total revenue for each country
country_revenue = df_sales.groupBy("Country").agg(
    sum("TotalSales").alias("TotalRevenue")
)

# Sort countries by revenue in descending order
country_revenue = country_revenue.orderBy(
    col("TotalRevenue").desc()
)

# Display the top 10 countries
country_revenue.show(10)

+--------------+------------------+
|       Country|      TotalRevenue|
+--------------+------------------+
|United Kingdom| 9001744.093998965|
|   Netherlands| 285446.3399999993|
|          EIRE| 283140.5199999992|
|       Germany| 228678.4000000002|
|        France|209625.37000000017|
|     Australia|138453.80999999997|
|         Spain|61558.560000000216|
|   Switzerland| 57067.60000000012|
|       Belgium| 41196.34000000001|
|        Sweden| 38367.83000000001|
+--------------+------------------+
only showing top 10 rows


Monthly Sales Analysis

In [ ]:
# Import the required functions
from pyspark.sql.functions import date_format, sum, col, to_timestamp

# Convert InvoiceDate string to timestamp first
df_sales = df_sales.withColumn(
    "InvoiceDate",
    to_timestamp(col("InvoiceDate"), "M/d/yy H:mm")
)

# Extract the month and year from InvoiceDate
monthly_sales = df_sales.withColumn(
    "SalesMonth",
    date_format(col("InvoiceDate"), "yyyy-MM")
)

# Calculate total sales for each month
monthly_sales = monthly_sales.groupBy("SalesMonth").agg(
    sum("TotalSales").alias("MonthlyRevenue")
)

# Sort the months in chronological order
monthly_sales = monthly_sales.orderBy("SalesMonth")

# Display monthly sales
monthly_sales.show(20, truncate=False)

+----------+------------------+
|SalesMonth|MonthlyRevenue    |
+----------+------------------+
|2010-12   |821452.73         |
|2011-01   |689811.6100000164 |
|2011-02   |522545.5600000113 |
|2011-03   |716215.2600000252 |
|2011-04   |536968.4910000095 |
|2011-05   |769296.6100000171 |
|2011-06   |760547.010000024  |
|2011-07   |718076.1210000236 |
|2011-08   |757841.3800000127 |
|2011-09   |1056435.1920000338|
|2011-10   |1151263.7300000424|
|2011-11   |1503866.7800000352|
|2011-12   |637790.3300000094 |
+----------+------------------+



# Phase 5: Data Validation
Ab tak humne:
- Duplicate records remove kiye.
- Cancelled transactions separate ki.
- Returns aur zero quantities separate ki.
- Invalid prices separate kiye.
- InvoiceDate ka data type check kiya.
- TotalSales column create kiya.

Ab check karenge ki hamara cleaned dataset properly prepared hai ya nahi.

Check missing values after cleaning

In [ ]:
# Import the required functions
from pyspark.sql.functions import col, sum

# Count missing values in each column
missing_values = df_sales.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_sales.columns
])

# Display missing value counts
missing_values.show()

# Display the data types of all columns
df_sales.printSchema()

# Display missing value counts
# missing_values.show()
# col – Column select karta hai.
# sum – Missing values ka total count calculate karta hai.
# Hum df_sales ke har column mein missing values count karne ke liye ek naya DataFrame bana rahe hain.

# col(c) – Current column select karo.
# isNull() – Missing value check karo.
# cast("int") – True ko 1 aur False ko 0 banao.
# sum() – Saare 1s add karo.
# alias(c) – Original column ka naam do.

+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|        0|        0|          0|       0|          0|        0|    132186|      0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)



Missing Values and Data Types Validation

In [ ]:
# Count total records in the sales dataset
total_records = df_sales.count()

# Count duplicate records
duplicate_records = total_records - df_sales.dropDuplicates().count()

# Count negative quantities
negative_quantities = df_sales.filter(
    col("Quantity") < 0
).count()

# Count zero or negative prices
invalid_prices = df_sales.filter(
    col("UnitPrice") <= 0
).count()

# Display validation results
print("Total Sales Records:", total_records)
print("Remaining Duplicates:", duplicate_records)
print("Negative Quantities:", negative_quantities)
print("Invalid Prices:", invalid_prices)

# df_sales.count()	Total sales records count karega
# dropDuplicates()	Duplicate rows remove karke count karega
# filter()	Condition ke according records find karega
# < 0	Negative quantities check karega
# <= 0	Zero ya negative prices check karega
# Hum ye checks kyun kar rahe hain?
# Humne cleaning stage mein duplicates, negative quantities aur invalid prices ko remove kiya tha.
# Ab verify kar rahe hain ki final sales dataset mein ye records actually nahi bache.

Total Sales Records: 524878
Remaining Duplicates: 0
Negative Quantities: 0
Invalid Prices: 0


Remove Records with Missing Essential Values karenge

In [ ]:
# Define the essential columns
required_columns = [
    "InvoiceNo",
    "StockCode",
    "Quantity",                 #Humne un columns ki list banayi hai jinmein valid information hona zaroori hai.
    "InvoiceDate",
    "UnitPrice"
]

# Count records before validation
records_before = df_sales.count()

# Remove records with missing essential values
df_sales = df_sales.dropna(
    subset=required_columns

    #dropna() – Missing values wale records remove karta hai.
    #subset – Batata hai ki kin specific columns mein missing values check karni hain.
)

# Count records after validation
records_after = df_sales.count()

# Display validation results
print("Records Before Validation:", records_before)
print("Records After Validation:", records_after)
print("Records Removed:", records_before - records_after)

Records Before Validation: 524878
Records After Validation: 524878
Records Removed: 0


Final Data Quality Check karenge.

In [ ]:
# Check missing values in essential columns
print("Missing InvoiceNo:", df_sales.filter(col("InvoiceNo").isNull()).count())

print("Missing StockCode:", df_sales.filter(col("StockCode").isNull()).count())

print("Missing Quantity:", df_sales.filter(col("Quantity").isNull()).count())

print("Missing InvoiceDate:", df_sales.filter(col("InvoiceDate").isNull()).count())

print("Missing UnitPrice:", df_sales.filter(col("UnitPrice").isNull()).count())

# Check whether TotalSales is missing
print("Missing TotalSales:", df_sales.filter(col("TotalSales").isNull()).count())

# Display final dataset dimensions
print("Final Rows:", df_sales.count())
print("Final Columns:", len(df_sales.columns))

# filter() – Condition ke according records select karta hai.
# isNull() – Check karta hai ki value missing hai ya nahi.
# count() – Matching records count karta hai.
# len(df_sales.columns) – Total columns count karta hai.

Missing InvoiceNo: 0
Missing StockCode: 0
Missing Quantity: 0
Missing InvoiceDate: 0
Missing UnitPrice: 0
Missing TotalSales: 0
Final Rows: 524878
Final Columns: 9
